# Exploratory Data Analysis
## Scam Message Risk Assistant — Milestone 1

**Purpose:** Characterise training data, visualise class distributions, analyse text-length and vocabulary patterns, and summarise the train / val / test splits.

**How to run:** Execute all cells top-to-bottom *after* running the data pipeline:
```bash
python -m src.data_prep
```

Charts in this notebook are publication-ready and suitable for a project report or presentation.

---
> **Domain reminder:** The binary label `1` covers both UCI 'spam' (includes marketing texts) and Kaggle 'Phishing Email'. They are merged for the baseline model but kept separate in the `source` column for per-domain analysis.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path
from collections import Counter

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')

# ── Project root ─────────────────────────────────────────────────────────────
# Works whether you run from project root or from notebooks/ folder
ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

# Consistent colours throughout the notebook
PALETTE = {0: '#4CAF50', 1: '#F44336'}          # green = ham, red = scam
LABEL_NAMES = {0: 'Legitimate (ham)', 1: 'Scam / Phishing'}

print('Setup complete.')
print(f'Project root : {ROOT}')

In [ ]:
# ── Load processed splits ─────────────────────────────────────────────────────
PROCESSED_DIR = ROOT / 'data' / 'processed'

def load_split(name: str) -> pd.DataFrame:
    path = PROCESSED_DIR / f'{name}.csv'
    assert path.exists(), (
        f"'{path}' not found.\n"
        f"Run the data pipeline first:\n"
        f"  python -m src.data_prep"
    )
    df = pd.read_csv(path)
    df['split'] = name
    return df

train_df = load_split('train')
val_df   = load_split('val')
test_df  = load_split('test')
full_df  = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f'Columns     : {list(full_df.columns)}')
print(f'Total rows  : {len(full_df):,}')
full_df.head(3)

---
## 1. Dataset Overview

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = (
    full_df.groupby('label')
    .agg(count=('text', 'count'))
    .rename(index=LABEL_NAMES)
)
summary['percentage'] = (summary['count'] / summary['count'].sum() * 100).round(1)
print('Class distribution across all splits:')
display(summary)

In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: counts
counts = full_df['label'].map(LABEL_NAMES).value_counts()
bars = axes[0].bar(
    counts.index,
    counts.values,
    color=[PALETTE[k] for k in full_df['label'].value_counts().index],
    edgecolor='white', linewidth=1.5
)
axes[0].set_title('Class Counts (full dataset)', fontweight='bold')
axes[0].set_ylabel('Number of messages')
axes[0].bar_label(bars, fmt='{:,.0f}', padding=4)
axes[0].set_ylim(0, max(counts.values) * 1.15)

# Right: percentage pie
axes[1].pie(
    counts.values,
    labels=counts.index,
    colors=[PALETTE[k] for k in full_df['label'].value_counts().index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Proportions', fontweight='bold')

plt.suptitle('Figure 1 — Class Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig1_class_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig1_class_distribution.png')

---
## 2. Data Source Breakdown

The `source` column lets us separate UCI SMS messages from Kaggle phishing emails.  
This is important because the two datasets represent **different threat types** — mixing them is a conscious trade-off for the baseline model.

In [ ]:
# ── Source × label cross-tab ──────────────────────────────────────────────────
source_tab = (
    full_df.groupby(['source', 'label'])
    .size()
    .unstack(fill_value=0)
    .rename(columns=LABEL_NAMES)
)
source_tab['Total'] = source_tab.sum(axis=1)
source_tab['Scam %'] = (
    source_tab[LABEL_NAMES[1]] / source_tab['Total'] * 100
).round(1)

print('Source breakdown:')
display(source_tab)

In [ ]:
# ── Stacked bar: source × class ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

plot_data = (
    full_df.groupby(['source', 'label'])
    .size()
    .unstack(fill_value=0)
)
plot_data.columns = [LABEL_NAMES[c] for c in plot_data.columns]
plot_data.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=[PALETTE[0], PALETTE[1]],
    edgecolor='white'
)
ax.set_title('Figure 2 — Message Count by Source and Class', fontweight='bold')
ax.set_xlabel('Dataset source')
ax.set_ylabel('Number of messages')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig2_source_breakdown.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig2_source_breakdown.png')

---
## 3. Text Length Analysis

Text length (in characters) often differs between legitimate messages and scam/phishing attempts.  
Phishing emails tend to be longer; SMS scams are constrained by character limits.

In [ ]:
# ── Compute character and word counts ─────────────────────────────────────────
full_df['char_len'] = full_df['text'].str.len()
full_df['word_len'] = full_df['text'].str.split().str.len()

# Summary statistics by class
len_stats = (
    full_df.groupby('label')[['char_len', 'word_len']]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
)
len_stats.index = [LABEL_NAMES[i] for i in len_stats.index]
print('Text length statistics by class:')
display(len_stats)

In [ ]:
# ── Length distribution plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label_val, label_name in LABEL_NAMES.items():
    subset = full_df[full_df['label'] == label_val]
    
    # Character length
    axes[0].hist(
        subset['char_len'].clip(upper=2000),
        bins=60, alpha=0.6,
        color=PALETTE[label_val], label=label_name
    )
    # Word count
    axes[1].hist(
        subset['word_len'].clip(upper=300),
        bins=60, alpha=0.6,
        color=PALETTE[label_val], label=label_name
    )

axes[0].set_title('Character Length Distribution', fontweight='bold')
axes[0].set_xlabel('Characters (clipped at 2 000)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Words (clipped at 300)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Figure 3 — Text Length by Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig3_text_length.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig3_text_length.png')

---
## 4. Top N-gram Analysis

Examining the most frequent unigrams and bigrams per class reveals the vocabulary patterns the TF-IDF model will learn to exploit.

In [ ]:
def top_ngrams(
    texts: pd.Series,
    ngram_range: tuple[int, int] = (1, 1),
    top_n: int = 20,
) -> pd.DataFrame:
    """
    Return a DataFrame of the top-n most frequent n-grams in *texts*,
    with stop-words removed.
    """
    vec = CountVectorizer(
        ngram_range=ngram_range,
        stop_words='english',
        max_features=5000,
    )
    X = vec.fit_transform(texts.dropna().astype(str))
    counts = X.sum(axis=0).A1
    vocab  = vec.get_feature_names_out()
    return (
        pd.DataFrame({'ngram': vocab, 'count': counts})
        .sort_values('count', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

In [ ]:
# ── Top 15 unigrams per class ─────────────────────────────────────────────────
ham_uni  = top_ngrams(full_df[full_df['label'] == 0]['text'], (1, 1), 15)
scam_uni = top_ngrams(full_df[full_df['label'] == 1]['text'], (1, 1), 15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, label_val, title in [
    (axes[0], ham_uni,  0, 'Top 15 Unigrams — Legitimate (ham)'),
    (axes[1], scam_uni, 1, 'Top 15 Unigrams — Scam / Phishing'),
]:
    bars = ax.barh(
        df['ngram'][::-1], df['count'][::-1],
        color=PALETTE[label_val], edgecolor='white'
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.bar_label(bars, fmt='{:,.0f}', padding=3)

plt.suptitle('Figure 4 — Top Unigrams by Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig4_top_unigrams.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig4_top_unigrams.png')

In [ ]:
# ── Top 15 bigrams per class ──────────────────────────────────────────────────
ham_bi  = top_ngrams(full_df[full_df['label'] == 0]['text'], (2, 2), 15)
scam_bi = top_ngrams(full_df[full_df['label'] == 1]['text'], (2, 2), 15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, label_val, title in [
    (axes[0], ham_bi,  0, 'Top 15 Bigrams — Legitimate (ham)'),
    (axes[1], scam_bi, 1, 'Top 15 Bigrams — Scam / Phishing'),
]:
    bars = ax.barh(
        df['ngram'][::-1], df['count'][::-1],
        color=PALETTE[label_val], edgecolor='white'
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.bar_label(bars, fmt='{:,.0f}', padding=3)

plt.suptitle('Figure 5 — Top Bigrams by Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig5_top_bigrams.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig5_top_bigrams.png')

---
## 5. Vocabulary Statistics

In [ ]:
# ── Vocabulary size and overlap ───────────────────────────────────────────────
def build_vocab(texts: pd.Series) -> set[str]:
    tokens = texts.dropna().str.split().explode()
    return set(tokens.str.strip().str.lower().dropna())

ham_vocab  = build_vocab(full_df[full_df['label'] == 0]['text'])
scam_vocab = build_vocab(full_df[full_df['label'] == 1]['text'])
shared     = ham_vocab & scam_vocab
ham_only   = ham_vocab - scam_vocab
scam_only  = scam_vocab - ham_vocab

vocab_stats = pd.DataFrame({
    'Metric': [
        'Ham vocabulary size',
        'Scam vocabulary size',
        'Shared tokens',
        'Ham-exclusive tokens',
        'Scam-exclusive tokens',
    ],
    'Count': [
        len(ham_vocab),
        len(scam_vocab),
        len(shared),
        len(ham_only),
        len(scam_only),
    ]
})
print('Vocabulary statistics:')
display(vocab_stats)

overlap_pct = len(shared) / len(ham_vocab | scam_vocab) * 100
print(f'\nVocabulary overlap: {overlap_pct:.1f}% of all unique tokens are shared.')
print('(Higher overlap → more challenging classification task)')

---
## 6. Train / Val / Test Split Summary

In [ ]:
# ── Split size and class balance table ────────────────────────────────────────
rows = []
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n_ham  = (df['label'] == 0).sum()
    n_scam = (df['label'] == 1).sum()
    rows.append({
        'Split' : name,
        'Total' : len(df),
        'Ham'   : n_ham,
        'Scam'  : n_scam,
        'Ham %' : round(n_ham  / len(df) * 100, 1),
        'Scam %': round(n_scam / len(df) * 100, 1),
    })
split_summary = pd.DataFrame(rows)
print('Train / val / test split summary:')
display(split_summary)

In [ ]:
# ── Split size chart ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x     = split_summary['Split']
bar_h = ax.bar(x, split_summary['Ham'],  label='Legitimate (ham)',  color=PALETTE[0], edgecolor='white')
bar_s = ax.bar(x, split_summary['Scam'], label='Scam / Phishing',   color=PALETTE[1], edgecolor='white',
               bottom=split_summary['Ham'])

ax.bar_label(bar_h, labels=[f"{v:,}" for v in split_summary['Ham']],  label_type='center', color='white', fontweight='bold')
ax.bar_label(bar_s, labels=[f"{v:,}" for v in split_summary['Scam']], label_type='center', color='white', fontweight='bold')

# Total label on top
for i, total in enumerate(split_summary['Total']):
    ax.text(i, total + total * 0.01, f'{total:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Figure 6 — Split Sizes and Class Balance (stratified 70/15/15)', fontweight='bold')
ax.set_ylabel('Number of messages')
ax.legend(loc='upper right')
ax.set_ylim(0, split_summary['Total'].max() * 1.12)

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig6_split_summary.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/fig6_split_summary.png')

---
## 7. Key Findings & EDA Summary

Run the cells above and fill in the numbers. Use this section directly in your project report.

| Finding | Value |
|---|---|
| Total messages | *(auto-fill from cell 3)* |
| Ham / Scam ratio | *(from class distribution table)* |
| Median ham message length (chars) | *(from length statistics)* |
| Median scam message length (chars) | *(from length statistics)* |
| Most common scam unigrams | *(from n-gram analysis)* |
| Vocabulary overlap | *(from vocabulary statistics)* |

### Implications for Model Design (Milestone 2)

- **Class imbalance?** → Use `class_weight='balanced'` in Logistic Regression.
- **Vocabulary overlap** → High overlap suggests TF-IDF *weighting* (not just presence) will matter; consider sublinear TF scaling.
- **Length difference** → Message length could serve as a simple supplementary feature alongside TF-IDF if it is strongly discriminative.
- **Source heterogeneity** → Track per-source metrics in evaluation to detect domain shift.

---
*Next: Milestone 2 — TF-IDF + Logistic Regression training & evaluation (`src/model.py`, `src/evaluation.py`)*